# GENA_LM All-Cell Benchmark Comparison

Compare GENA_LM predictions with the full qnorm GT matrix. The GT matrix is cell x gene and is transformed to gene x cell.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO = Path.cwd()
if not (REPO / "downstream_tasks").exists():
    REPO = Path("/home/jovyan/dpanc/GENA_LM/GENA_LM_expression_branch")

BENCHMARK_ROOT = Path("/home/jovyan/dpanc/benchmarking")
DATA = BENCHMARK_ROOT / "data"
GENA_ROOT = BENCHMARK_ROOT / "GENA_LM"
ALPHAGENOME_ROOT = BENCHMARK_ROOT / "AlphaGenome"

sys.path.insert(0, str(REPO / "downstream_tasks/expression_prediction/gena_lm_benchmark/scripts"))
from score_ct_specificity import score_predictions


In [ ]:
model = "full_model"
split = "valid"      # valid or test
cell_set = "json812" # json14 or json812

pred_path = GENA_ROOT / "predictions_results" / model / f"gena_lm_{split}_{cell_set}_predictions.csv"
true_path = DATA / "borzoi_all_ids_qnorm_matrix.csv"
selected_targets_path = DATA / "selected_targets.csv"
out_dir = GENA_ROOT / "comparison_results" / model
out_dir.mkdir(parents=True, exist_ok=True)

pred_raw = pd.read_csv(pred_path)
true_raw = pd.read_csv(true_path)

if "gene_id" not in pred_raw.columns:
    pred_raw = pred_raw.rename(columns={pred_raw.columns[0]: "gene_id"})

metadata_cols = ["gene_id", "id", "original_id", "targets_identifier_base", "strand_specificity"]
metadata_cols = [c for c in metadata_cols if c in true_raw.columns]
gene_cols = [c for c in true_raw.columns if c not in metadata_cols]
cell_id_col = "gene_id" if "gene_id" in true_raw.columns else "id"

true_like_prediction = true_raw.set_index(cell_id_col)[gene_cols].T.reset_index().rename(columns={"index": "gene_id"})

print("pred_raw:", pred_raw.shape)
print("true_raw:", true_raw.shape)
print("true_like_prediction:", true_like_prediction.shape)
display(pred_raw.head())
display(true_like_prediction.head())


In [ ]:
def correlation_table(true_df, pred_df):
    true = true_df.set_index("gene_id")
    pred = pred_df.set_index("gene_id")

    common_genes = true.index.intersection(pred.index)
    common_cells = true.columns.intersection(pred.columns)

    true = true.loc[common_genes, common_cells]
    pred = pred.loc[common_genes, common_cells]

    rows = []
    for cell in common_cells:
        true_vec = true[cell].astype(float).values
        pred_vec = pred[cell].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) < 2 or np.std(true_vec) == 0 or np.std(pred_vec) == 0:
            corr = np.nan
        else:
            corr = np.corrcoef(true_vec, pred_vec)[0, 1]
        rows.append({"cell_type": cell, "corr_genes": corr})

    result = pd.DataFrame(rows)

    gene_corrs = []
    skipped_genes = []
    for gene in common_genes:
        true_vec = true.loc[gene].astype(float).values
        pred_vec = pred.loc[gene].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) < 4 or np.std(true_vec) == 0 or np.std(pred_vec) == 0:
            skipped_genes.append(gene)
            continue
        corr = np.corrcoef(true_vec, pred_vec)[0, 1]
        if np.isfinite(corr):
            gene_corrs.append(corr)
        else:
            skipped_genes.append(gene)

    mean_corr_cells = float(np.mean(gene_corrs)) if gene_corrs else np.nan
    result["corr_cells"] = mean_corr_cells

    mean_row = pd.DataFrame([{
        "cell_type": "mean",
        "corr_genes": result["corr_genes"].mean(),
        "corr_cells": mean_corr_cells,
    }])
    result = pd.concat([result, mean_row], ignore_index=True)

    return result, true.reset_index(), pred.reset_index(), common_genes, common_cells, skipped_genes


In [ ]:
result, true_aligned, pred_aligned, common_genes, common_cells, skipped_genes = correlation_table(true_like_prediction, pred_raw)
print("common genes:", len(common_genes))
print("common cells:", len(common_cells))
print("skipped genes for corr_cells:", len(skipped_genes))
display(result)


In [ ]:
score_dict = score_predictions(true_aligned, pred_aligned, str(selected_targets_path), need_log=False)
deviation_r = score_dict.get("deviation_r", float("nan"))
print("deviation_r:", deviation_r)
score_dict


In [ ]:
result_path = out_dir / f"correlations_{split}_{cell_set}.csv"
summary_path = out_dir / f"summary_{split}_{cell_set}.csv"

result.to_csv(result_path, index=False)
summary = pd.DataFrame([{
    "model": model,
    "split": split,
    "cell_set": cell_set,
    "genes": len(common_genes),
    "cells": len(common_cells),
    "corr_genes_mean": result.loc[result["cell_type"] != "mean", "corr_genes"].mean(),
    "corr_cells_mean": result.loc[result["cell_type"] == "mean", "corr_cells"].iloc[0],
    "deviation_r": deviation_r,
}])
summary.to_csv(summary_path, index=False)
print("saved:", result_path)
print("saved:", summary_path)
display(summary)
